# SuperWASP Pulsator Detection — Notebook 1: Data Download

**Author:** Desislava Lebessis  
**Dataset:** Zenodo VeSPA — McMaster et al. 2021, Res. Notes AAS 5 228  
**DOI:** [10.5281/zenodo.14937227](https://doi.org/10.5281/zenodo.14937227)

**What this notebook does:**
1. Downloads the VeSPA classification CSV from Zenodo (137 MB)
2. Filters pulsators and non-pulsators
3. Downloads light curve images to Google Drive
4. Saves a checkpoint so downloads can be resumed if interrupted

**Google Drive folder structure:**
```
SuperWASP/
    pulsatoren/        ← pulsator images
    non_pulsatoren/    ← non-pulsator images
    export.csv
    checkpoint.json
```

> **Next:** Run `superwasp_02_training.ipynb` to train the model.

---
## STEP 1: Install libraries

In [ ]:
!pip install requests pandas Pillow tqdm -q

import os
import json
import time
import random
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from google.colab import drive
import matplotlib.pyplot as plt

print('✅ Libraries loaded!')

---
## STEP 2: Connect Google Drive and set paths

In [ ]:
drive.mount('/content/drive')

# ═══════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════
BASIS_PFAD        = '/content/drive/MyDrive/SuperWASP'
MAX_NON_PULS      = 2000   # Maximum non-pulsators
PAUSE             = 0.1    # Pause between downloads
# ═══════════════════════════════════════════

PULS_PFAD         = os.path.join(BASIS_PFAD, 'pulsatoren')
NON_PULS_PFAD     = os.path.join(BASIS_PFAD, 'non_pulsatoren')
CHECKPOINT_PFAD   = os.path.join(BASIS_PFAD, 'checkpoint.json')
CSV_PFAD          = os.path.join(BASIS_PFAD, 'export.csv')

os.makedirs(PULS_PFAD, exist_ok=True)
os.makedirs(NON_PULS_PFAD, exist_ok=True)

print('✅ Google Drive connected!')
print(f'   Pulsator folder:     {PULS_PFAD}')
print(f'   Non-pulsator folder: {NON_PULS_PFAD}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

---
## STEP 3: Download VeSPA classification CSV from Zenodo

In [ ]:
# Download CSV directly from Zenodo
ZENODO_URL = 'https://zenodo.org/records/14937227/files/export.csv'

if os.path.exists(CSV_PFAD):
    print('✅ CSV already exists — skipping download')
else:
    print('🔄 Downloading CSV from Zenodo...')
    print('   This takes 2-5 minutes (137 MB)...')

    antwort = requests.get(ZENODO_URL, stream=True)
    gesamt  = int(antwort.headers.get('content-length', 0))

    with open(CSV_PFAD, 'wb') as f:
        heruntergeladen = 0
        for chunk in antwort.iter_content(chunk_size=8192):
            f.write(chunk)
            heruntergeladen += len(chunk)
            prozent = heruntergeladen / gesamt * 100 if gesamt > 0 else 0
            if heruntergeladen % (1024*1024*10) < 8192:
                print(f'   {prozent:.0f}% — {heruntergeladen/1024/1024:.0f} MB')

    print(f'\n✅ CSV downloaded: {CSV_PFAD}')

---
## STEP 4: Load and analyze the CSV

In [ ]:
# Load CSV in chunks to save RAM
print('🔄 Loading CSV...')

chunks = []
for chunk in pd.read_csv(CSV_PFAD, chunksize=10000, low_memory=False):
    chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)

print(f'✅ CSV loaded!')
print(f'   Total rows: {len(df):,}')
print(f'   Columns: {list(df.columns)}')
print(f'\n   First 3 rows:')
print(df.head(3).to_string())

In [ ]:
# Show classifications
klassen_spalte = 'Classification'
url_spalte = 'Folded plot URL'

print(f'✅ Columns found!')
print(f'   Classification: {klassen_spalte}')
print(f'   Image URL: {url_spalte}')
print(f'\n   All classes:')
verteilung = df[klassen_spalte].value_counts()
print(verteilung.to_string())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
verteilung.plot(kind='bar', ax=ax, color='#2E75B6')
ax.set_title('Class distribution in Zenodo dataset', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## STEP 5: Find classification column and analyze class distribution

In [ ]:
# Find classification column
# Possible column names
moegliche_klassen_spalten = [
    'classification', 'class', 'type', 'category',
    'label', 'subject_type', 'period_flag', 'best_classification'
]

klassen_spalte = None
for spalte in moegliche_klassen_spalten:
    if spalte in df.columns:
        klassen_spalte = spalte
        break

if klassen_spalte:
    print(f'✅ Classification column found: "{klassen_spalte}"')
    print(f'\n   All classes in dataset:')
    verteilung = df[klassen_spalte].value_counts()
    print(verteilung.to_string())

    # Visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    verteilung.plot(kind='bar', ax=ax, color='#2E75B6')
    ax.set_title('Class distribution in Zenodo dataset', fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(BASIS_PFAD, 'klassenverteilung.png'), dpi=150)
    plt.show()
else:
    print('⚠️ Classification column not found!')
    print('   Available columns:', list(df.columns))
    print('   Please identify the correct column manually.')

---
## STEP 6: Filter pulsators and non-pulsators

In [ ]:
# ═══════════════════════════════════════════
# ADJUST HERE if needed
# ═══════════════════════════════════════════
KLASSEN_SPALTE    = klassen_spalte   # Auto-detected in Step 5
PULSATOR_BEGRIFF  = 'PULSATOR'       # Exact keyword from CSV
# ═══════════════════════════════════════════

# Filter pulsators — case-insensitive
pulsatoren_df = df[
    df[KLASSEN_SPALTE].str.upper().str.contains('PULSATOR', na=False)
]

# Filter non-pulsators — everything else
non_puls_df = df[
    ~df[KLASSEN_SPALTE].str.upper().str.contains('PULSATOR', na=False)
]

# Select 2000 non-pulsators — balanced across all classes
non_puls_klassen = non_puls_df[KLASSEN_SPALTE].unique()
pro_klasse = MAX_NON_PULS // len(non_puls_klassen)

non_puls_auswahl = []
for klasse in non_puls_klassen:
    klasse_df = non_puls_df[non_puls_df[KLASSEN_SPALTE] == klasse]
    auswahl   = klasse_df.sample(min(pro_klasse, len(klasse_df)), random_state=42)
    non_puls_auswahl.append(auswahl)

non_puls_final = pd.concat(non_puls_auswahl, ignore_index=True)

# Fill up to 2000 if needed
if len(non_puls_final) < MAX_NON_PULS:
    rest = non_puls_df.sample(
        min(MAX_NON_PULS - len(non_puls_final), len(non_puls_df)),
        random_state=42
    )
    non_puls_final = pd.concat([non_puls_final, rest], ignore_index=True)
    non_puls_final = non_puls_final.drop_duplicates()

print(f'✅ Selection complete!')
print(f'   Pulsators found:        {len(pulsatoren_df):4d}')
print(f'   Non-pulsators selected: {len(non_puls_final):4d}')
print(f'   Total:                  {len(pulsatoren_df) + len(non_puls_final):4d}')
print(f'\n   Non-pulsator class breakdown:')
print(non_puls_final[KLASSEN_SPALTE].value_counts().to_string())

In [ ]:
# Limit pulsators to 2000
MAX_PULSATOREN = 2000

pulsatoren_df = pulsatoren_df.sample(
    min(MAX_PULSATOREN, len(pulsatoren_df)),
    random_state=42
)

print(f'✅ Pulsators limited to: {len(pulsatoren_df)}')
print(f'   Non-pulsators:       {len(non_puls_final)}')
print(f'   Total:               {len(pulsatoren_df) + len(non_puls_final)}')

---
## STEP 7: Download light curve images

In [ ]:
import json
import time
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm

# Load checkpoint if available
def checkpoint_laden():
    if os.path.exists(CHECKPOINT_PFAD):
        with open(CHECKPOINT_PFAD, 'r') as f:
            return json.load(f)
    return {'heruntergeladen': [], 'fehler': []}

def checkpoint_speichern(daten):
    with open(CHECKPOINT_PFAD, 'w') as f:
        json.dump(daten, f)

def bild_herunterladen(url, pfad, dateiname):
    ziel = os.path.join(pfad, dateiname)
    if os.path.exists(ziel):
        return True
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            bild = Image.open(BytesIO(r.content)).convert('RGB')
            if bild.width >= 50 and bild.height >= 50:
                bild.save(ziel, quality=95)
                return True
    except Exception:
        pass
    return False

# Load checkpoint
checkpoint = checkpoint_laden()
bereits_heruntergeladen = set(checkpoint['heruntergeladen'])

# Prepare all downloads
alle_downloads = []

for _, zeile in pulsatoren_df.iterrows():
    if pd.notna(zeile.get(url_spalte)):
        alle_downloads.append({
            'url':    zeile[url_spalte],
            'ordner': PULS_PFAD,
            'klasse': 'pulsator'
        })

for _, zeile in non_puls_final.iterrows():
    if pd.notna(zeile.get(url_spalte)):
        alle_downloads.append({
            'url':    zeile[url_spalte],
            'ordner': NON_PULS_PFAD,
            'klasse': 'non_pulsator'
        })

print(f'📥 Download starts...')
print(f'   Total images:    {len(alle_downloads)}')
print(f'   Already present: {len(bereits_heruntergeladen)}')

neu_puls = neu_non = fehler = 0

for i, item in enumerate(tqdm(alle_downloads, desc='Download')):
    url       = item['url']
    ordner    = item['ordner']
    dateiname = f"superwasp_{i:05d}.jpg"
    key       = f"{ordner}/{dateiname}"

    if key in bereits_heruntergeladen:
        continue

    erfolg = bild_herunterladen(url, ordner, dateiname)

    if erfolg:
        bereits_heruntergeladen.add(key)
        if item['klasse'] == 'pulsator':
            neu_puls += 1
        else:
            neu_non += 1
    else:
        fehler += 1

    # Save checkpoint every 50 images
    if (neu_puls + neu_non) % 50 == 0 and (neu_puls + neu_non) > 0:
        checkpoint['heruntergeladen'] = list(bereits_heruntergeladen)
        checkpoint_speichern(checkpoint)

    time.sleep(0.1)

# Save final checkpoint
checkpoint['heruntergeladen'] = list(bereits_heruntergeladen)
checkpoint_speichern(checkpoint)

final_p = len(os.listdir(PULS_PFAD))
final_n = len(os.listdir(NON_PULS_PFAD))

print(f'\n✅ Download complete!')
print(f'   Pulsators:     {final_p:4d} images')
print(f'   Non-pulsators: {final_n:4d} images')
print(f'   Total:         {final_p + final_n:4d} images')
print(f'   Errors:        {fehler:4d}')

---
## STEP 8: Dataset overview

In [ ]:
# Show overview
final_p = len(os.listdir(PULS_PFAD))
final_n = len(os.listdir(NON_PULS_PFAD))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SuperWASP Dataset — Overview', fontsize=13, fontweight='bold')

klassen  = ['Pulsators', 'Non-Pulsators']
anzahlen = [final_p, final_n]
farben   = ['#C9A227', '#2E75B6']

bars = ax1.bar(klassen, anzahlen, color=farben, width=0.5)
ax1.set_title('Image count')
ax1.set_ylabel('Count')
for bar, anzahl in zip(bars, anzahlen):
    ax1.text(bar.get_x() + bar.get_width()/2.,
             bar.get_height() + 1, str(anzahl),
             ha='center', fontweight='bold')

ax2.pie(anzahlen, labels=klassen, colors=farben,
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Distribution')

plt.tight_layout()
plt.savefig(os.path.join(BASIS_PFAD, 'datensatz_uebersicht.png'), dpi=150)
plt.show()

print(f'\n📊 Your dataset is ready!')
print(f'   Pulsators:     {final_p}')
print(f'   Non-pulsators: {final_n}')
print(f'   Total:         {final_p + final_n}')
print(f'\n✅ All saved to Google Drive under: SuperWASP/')
print(f'\n→ Next step: Open Notebook 2 to train the model!')

---
## Done — Notebook 1 complete!

**Saved to Google Drive:**
- `SuperWASP/pulsatoren/` — pulsator light curve images
- `SuperWASP/non_pulsatoren/` — non-pulsator light curve images
- `SuperWASP/export.csv` — VeSPA classification data
- `SuperWASP/checkpoint.json` — download progress

**Citation:**  
McMaster et al. 2021, Res. Notes AAS 5 228. DOI: 10.5281/zenodo.14937227

**Next:** Open `superwasp_02_training.ipynb` to train the ResNet18 model.